In [13]:
import pandas as pd
import json
import re
import os

In [14]:
with open('../data/extracted_jobs_result_wanted.json', 'r', encoding='utf-8') as f: wanted = json.load(f)
with open('../data/extracted_jobs_result_jobkorea.json', 'r', encoding='utf-8') as f: jobkorea = json.load(f)
with open('../data/extracted_jobs_result_saramin.json', 'r', encoding='utf-8') as f: saramin = json.load(f)

df_wanted = pd.DataFrame(wanted)
df_jobkorea = pd.DataFrame(jobkorea)
df_saramin = pd.DataFrame(saramin)

print(f"원티드 데이터 개수: {len(df_wanted)}개 / 컬럼: {list(df_wanted.columns)}")
print(f"잡코리아 데이터 개수: {len(df_jobkorea)}개 / 컬럼: {list(df_jobkorea.columns)}")
print(f"사람인 데이터 개수: {len(df_saramin)}개 / 컬럼: {list(df_saramin.columns)}")

원티드 데이터 개수: 4404개 / 컬럼: ['job_id', 'tag_id', 'year_filter', 'company', 'position', 'main_tasks', 'requirements', 'preferred', 'skill_tags', 'location', 'scraped_date', 'hard_skills', 'soft_skills', 'preferences', 'culture_keywords', 'urgency_score', 'urgency_reason']
잡코리아 데이터 개수: 10107개 / 컬럼: ['공고번호', '회사명', '공고제목', '연차/경력', '지역', '기술스택/분야', '마감일', '상세내용', 'hard_skills', 'soft_skills', 'preferences', 'culture_keywords', 'urgency_score', 'urgency_reason']
사람인 데이터 개수: 12602개 / 컬럼: ['공고번호', '원문', 'hard_skills', 'soft_skills', 'preferences', 'culture_keywords', 'urgency_score', 'urgency_reason']


In [15]:
# urgency_score 샘플 형태 확인
print("[원티드] urgency_score 샘플:")
if 'urgency_score' in df_wanted.columns:
    print(df_wanted['urgency_score'].unique()[:10])
else:
    print("'urgency_score' 컬럼이 원티드에 없습니다.")

print("\n[잡코리아] urgency_score 샘플:")
if 'urgency_score' in df_jobkorea.columns:
    print(df_jobkorea['urgency_score'].unique()[:10])
else:
    print("'urgency_score' 컬럼이 잡코리아에 없습니다.")

print("\n[사람인] urgency_score 샘플:")
if 'urgency_score' in df_saramin.columns:
    print(df_saramin['urgency_score'].unique()[:10])
else:
    print("'urgency_score' 컬럼이 사람인에 없습니다.")

[원티드] urgency_score 샘플:
[4 3 2]

[잡코리아] urgency_score 샘플:
[3 2 1 4 5]

[사람인] urgency_score 샘플:
[3 4 1 2 5]


In [16]:
print()
if 'urgency_score' in df_wanted.columns:
    print(df_wanted['urgency_score'].value_counts().sort_index())


urgency_score
2       4
3    2269
4    2131
Name: count, dtype: int64


In [17]:
# 특수문자, 공백, 이메일 등 제거
def clean_for_model(text):
    if not text or pd.isna(text):
        return ""
    # URL, 이메일 주소 제거
    text = re.sub(r'http\S+|www\S+|<[^>]*>', ' ', str(text))
    text = re.sub(r'\S+@\S+', ' ', text)
    # 한글, 영어, 숫자, 기본 문장부호만 유지
    text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?~]', ' ', text)
    # 연속된 공백 하나로 통합
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [18]:
# 원티드 전처리(position + main_tasks + requirements + preferred)
def process_wanted(df):
    rows = []
    df_valid_data = df[df['urgency_score'].notna()].copy()
    
    for _, row in df_valid_data.iterrows():
        full_text = f"{row.get('position', '')} {row.get('main_tasks', '')} {row.get('requirements', '')} {row.get('preferred', '')}"
        
        rows.append({
            'text': clean_for_model(full_text),
            'label': int(float(row['urgency_score'])),
            'platform': 'wanted'
        })
    return pd.DataFrame(rows)

In [19]:
# 잡코리아 전처리(공고제목 + 상세내용)
def process_jobkorea(df):
    rows = []
    df_valid_data = df[df['urgency_score'].notna()].copy()
    
    for _, row in df_valid_data.iterrows():
        full_text = f"{row.get('공고제목', '')} {row.get('상세내용', '')}"
        
        rows.append({
            'text': clean_for_model(full_text),
            'label': int(float(row['urgency_score'])),
            'platform': 'jobKorea'
        })
    return pd.DataFrame(rows)

In [20]:
# 사람인 전처리(원문)
def process_jobkorea(df):
    rows = []
    df_valid_data = df[df['urgency_score'].notna()].copy()
    
    for _, row in df_valid_data.iterrows():
        full_text = f"{row.get('원문', '')}"
        
        rows.append({
            'text': clean_for_model(full_text),
            'label': int(float(row['urgency_score'])),
            'platform': 'jobKorea'
        })
    return pd.DataFrame(rows)

In [22]:
# 전처리 실행
df_wanted_processed = process_wanted(df_wanted)
df_jobkorea_processed = process_jobkorea(df_jobkorea)
df_saramin_processed = process_jobkorea(df_saramin)

print(f'원티드 전처리 완료: {len(df_wanted_processed)}건')
print(f'잡코리아 전처리 완료: {len(df_jobkorea_processed)}건')
print(f'사람인 전처리 완료: {len(df_saramin_processed)}건')

원티드 전처리 완료: 4404건
잡코리아 전처리 완료: 10107건
사람인 전처리 완료: 12602건


In [29]:
# 데이터셋 하나로 합치기
df_combine_all = pd.concat([df_wanted_processed, df_jobkorea_processed, df_saramin_processed], ignore_index=True)

# 중복 데이터 제거
df_combine_all = df_combine_all.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f'[최종 완성] 머신러닝 학습 모델에 들어갈 통합 데이터 개수: {len(df_combine_all)}개')
print('\n통합 데이터셋 내 플랫폼별 지분:')
print(df_combine_all['platform'].value_counts())

print('\n통합 데이터셋 내 라벨별 지분:')
print(df_combine_all['label'].value_counts().sort_index())

[최종 완성] 머신러닝 학습 모델에 들어갈 통합 데이터 개수: 16053개

통합 데이터셋 내 플랫폼별 지분:
platform
jobKorea    12355
wanted       3698
Name: count, dtype: int64

통합 데이터셋 내 라벨별 지분:
label
1      107
2      194
3    11604
4     4130
5       18
Name: count, dtype: int64


In [ ]:
OUTPUT_FILE = './data/integrated_learning_dataset.csv'

df_combine_all.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'{OUTPUT_FILE}로 통합 데이터 저장 완료')